# 04: Combined Recruitment Signal
**Goal:** combine the value forecast (notebook 02) and risk score (notebook 03) into one recruitment signal table, and export it for the Tableau dashboard.

**Important caveat carried forward from notebook 02:** `trend_direction` has a confirmed systematic bias toward 'rising' (only 22.7% directional accuracy in backtesting, worse than a 50% coin flip). This notebook uses it anyway since it's the project's core signal, but the quadrant counts below will be skewed toward 'rising' as a direct result -- this needs to be stated explicitly in the presentation, not presented as if the model reliably identifies who's actually rising in value.

## Setup

In [1]:
import pandas as pd
import numpy as np

## Load both notebook outputs

In [2]:
value_forecasts = pd.read_csv('../data/processed/value_forecasts.csv')
risk_scores = pd.read_csv('../data/processed/risk_scores.csv')

print(f'{len(value_forecasts)} players with value forecasts')
print(f'{len(risk_scores)} players with risk scores')

signal = value_forecasts.merge(
    risk_scores[['player_id', 'injury_count', 'total_days_missed',
                  'total_90s_played', 'risk_score']],
    on='player_id', how='inner'
)
print(f'{len(signal)} players with both')
signal.head()

1518 players with value forecasts
1959 players with risk scores
1518 players with both


,player_id,canonical_name,current_value,forecast_6mo,trend_direction,forecast_6mo_raw,injury_count,total_days_missed,total_90s_played,risk_score
0,132,Tomas Rosicky,350000.0,7.100304e+05,rising,7.100304e+05,16,1301.0,0.0,0.260862
1,488,Gerhard Tremmel,250000.0,5.090490e+05,rising,5.090490e+05,0,0.0,0.0,0.000000
2,1397,Michael Owen,1000000.0,2.003874e+06,rising,2.003874e+06,9,868.0,0.0,0.174047
3,1573,Thomas Hitzlsperger,1000000.0,2.902341e+06,rising,2.902341e+06,6,417.0,0.0,0.166152
4,2514,Bastian Schweinsteiger,1000000.0,3.000000e+06,rising,7.622494e+06,31,641.0,0.0,0.416640


## Flag the known injury-count cap
Carried forward from notebook 03 -- players hitting the exact 60-injury data cap tie at the maximum frequency score, which can inflate their risk ranking beyond their real relative risk. Keeping this visible in the final output rather than hiding it.

In [3]:
signal['hit_injury_cap'] = signal['injury_count'] == 60
print(f"{signal['hit_injury_cap'].sum()} / {len(signal)} players in the "
      f"final signal hit the injury-count cap")

12 / 1518 players in the final signal hit the injury-count cap


## Define the recruitment signal
Simple, explainable quadrants rather than a single opaque number -- this maps directly onto how a scouting department would actually think about it:
- **Priority target:** rising value forecast + low risk
- **Caution flag:** rising value forecast + high risk (the mispriced group the original hypothesis is about)
- **Low priority:** flat/declining value forecast, regardless of risk

**Risk threshold:** using the median risk_score as the high/low cutoff -- a principled, data-driven split rather than an arbitrary fixed number, and it keeps the high/low risk groups balanced in size.

In [4]:
risk_threshold = signal['risk_score'].median()
print(f'Risk threshold (median): {risk_threshold:.3f}')

def classify(row):
    rising = row['trend_direction'] == 'rising'
    high_risk = row['risk_score'] > risk_threshold
    if rising and not high_risk:
        return 'Priority target'
    elif rising and high_risk:
        return 'Caution flag'
    else:
        return 'Low priority'

signal['recruitment_signal'] = signal.apply(classify, axis=1)
signal['recruitment_signal'].value_counts()

Risk threshold (median): 0.241


recruitment_signal
Caution flag       726
Priority target    685
Low priority       107
Name: count, dtype: int64

## Sanity-check the distribution
Given the known 'rising' bias, expect Priority target + Caution flag to together outnumber Low priority by more than the true underlying rate would justify. Worth stating this plainly rather than presenting the raw counts as a trustworthy split.

In [5]:
n_rising = (signal['trend_direction'] == 'rising').sum()
print(f"{n_rising} / {len(signal)} players ({n_rising/len(signal):.0%}) "
      f"flagged as 'rising' -- compare against the ~90% over-prediction "
      f"rate found in notebook 02's backtest")

1411 / 1518 players (93%) flagged as 'rising' -- compare against the ~90% over-prediction rate found in notebook 02's backtest


## Look at the Caution flag group specifically
This is the group the original hypothesis is actually about: players the market may be under-pricing for risk. Worth inspecting by name, not just counting.

In [6]:
caution = signal[signal['recruitment_signal'] == 'Caution flag'].sort_values(
    'risk_score', ascending=False)
caution[['canonical_name', 'current_value', 'forecast_6mo', 'risk_score',
         'hit_injury_cap']].head(20)

,canonical_name,current_value,forecast_6mo,risk_score,hit_injury_cap
522,Aaron Cresswell,450000.0,1.350000e+06,0.815522,True
661,Divock Origi,5000000.0,1.500000e+07,0.694165,True
640,Fabian Schär,4000000.0,1.200000e+07,0.693314,True
1176,Riccardo Calafiori,55000000.0,1.146721e+08,0.679811,True
379,Chris Löwe,150000.0,4.500000e+05,0.674158,True
282,Shane Long,250000.0,7.500000e+05,0.667916,True
141,Phil Jagielka,200000.0,5.734785e+05,0.632373,True
1041,Jordan Beyer,1500000.0,4.500000e+06,0.631461,True
92,Petr Cech,2500000.0,7.376392e+06,0.624914,True
629,Jordan Pickford,13000000.0,3.900000e+07,0.606784,False


In [7]:
print(signal[signal['canonical_name'] == 'Petr Cech'][
    ['canonical_name', 'current_value', 'forecast_6mo', 'forecast_6mo_raw', 'trend_direction', 'recruitment_signal']
])

# Also check how many rows have this same contradiction
inconsistent = signal[
    ((signal['forecast_6mo'] > signal['current_value']) != (signal['trend_direction'] == 'rising'))
]
print(f'\n{len(inconsistent)} rows where forecast_6mo and trend_direction disagree')

   canonical_name  current_value  forecast_6mo  forecast_6mo_raw  \
92      Petr Cech      2500000.0  7.376392e+06      7.376392e+06   

   trend_direction recruitment_signal  
92          rising       Caution flag  

0 rows where forecast_6mo and trend_direction disagree


## Export for Tableau

In [8]:
import os
os.makedirs('../data/processed', exist_ok=True)
signal.to_csv('../data/processed/recruitment_signal.csv', index=False)
print(f'Exported {len(signal)} players to recruitment_signal.csv')

Exported 1518 players to recruitment_signal.csv


In [1]:
import pandas as pd

signal = pd.read_csv('../data/processed/recruitment_signal.csv')

# Round the noisy float columns to 2 decimals -- cleans up both the display
# ugliness and any locale/parsing edge cases from long decimal tails
for col in ['current_value', 'forecast_6mo', 'forecast_6mo_raw',
            'total_days_missed', 'total_90s_played', 'risk_score']:
    signal[col] = signal[col].round(2)

signal.to_csv('../data/processed/recruitment_signal_clean.csv', index=False)
print('Saved cleaned CSV')

Saved cleaned CSV


In [2]:
import pandas as pd
signal = pd.read_csv('../data/processed/recruitment_signal_clean.csv')  # or whichever final file
matches = signal[signal['canonical_name'] == 'Aaron Connolly']
print(matches[['player_id', 'canonical_name', 'current_value', 'forecast_6mo', 'risk_score']])

      player_id  canonical_name  current_value  forecast_6mo  risk_score
1095     434207  Aaron Connolly      3500000.0     5226972.7        0.35


In [3]:
signal = pd.read_csv('../data/processed/recruitment_signal_clean.csv')  # your final file
check = signal[signal['canonical_name'].isin(['Mohamed Salah', 'İlkay Gündoğan', 'Granit Xhaka', 'Virgil van Dijk'])]
print(check[['canonical_name', 'current_value', 'forecast_6mo', 'hit_injury_cap']])

      canonical_name  current_value  forecast_6mo  hit_injury_cap
373   İlkay Gündoğan      2000000.0     6000000.0           False
565     Granit Xhaka      8000000.0    24000000.0           False
644  Virgil van Dijk     15000000.0    45000000.0           False
662    Mohamed Salah     22000000.0    66000000.0           False


In [4]:
signal = pd.read_csv('../data/processed/recruitment_signal_clean.csv')
haaland = signal[signal['canonical_name'] == 'Erling Haaland']
print(haaland[['canonical_name', 'current_value', 'forecast_6mo', 'hit_injury_cap']])

      canonical_name  current_value  forecast_6mo  hit_injury_cap
1065  Erling Haaland    200000000.0  4.851652e+08           False


## Notes for the presentation
- State the 'rising' directional bias explicitly whenever quadrant counts are shown -- don't let the audience assume 'Priority target' means 'confirmed rising', since the underlying forecast has a known bias.
- The 'Caution flag' players -- do any match real, well-known transfer sagas where value and fitness were both genuinely in question? Naming a couple of specific, verifiable examples is stronger than the aggregate count alone.
- This quadrant framing mirrors how you'd have evaluated candidate risk vs. potential in HR recruitment -- a good place to draw that connection explicitly in the presentation.
- Flag the injury-cap and directional-bias limitations together as the project's two main honestly-documented weaknesses, rather than letting either surface as a surprise during Q&A.